In [8]:
from pathlib import Path

import pandas as pd
import soundfile as sf

DATASET_ROOT = Path('/home/uriel/Desktop/BSc/GenAI_Workboard/DnG/jan-2026-dl-gen-ai-project/messy_mashup')
GENRES_STEMS = DATASET_ROOT / 'genres_stems'
TEST_CSV = DATASET_ROOT / 'test.csv'
NOISE_DIR = DATASET_ROOT / 'ESC-50-master' / 'audio'

## 1. Train Balance

The training set is nicely balanced: every genre contributes 100 songs. That means class imbalance is not really the story here. The real challenge is that we will train on clean stems and then evaluate on noisy, already-mixed test audio.


In [9]:
genre_rows = []
for genre_dir in sorted(path for path in GENRES_STEMS.iterdir() if path.is_dir()):
    song_count = sum(1 for path in genre_dir.iterdir() if path.is_dir())
    genre_rows.append({'genre': genre_dir.name, 'num_songs': song_count})

genre_df = pd.DataFrame(genre_rows).sort_values('genre').reset_index(drop=True)
display(genre_df)
print('Total songs:', int(genre_df['num_songs'].sum()))


,genre,num_songs
0,blues,100
1,classical,100
2,country,100
3,disco,100
4,hiphop,100
5,jazz,100
6,metal,100
7,pop,100
8,reggae,100
9,rock,100


Total songs: 1000


## 2. Train type

The training side is very clean and regular. The stems are stereo, sampled at 44.1 kHz, and sit very close to 30 seconds in duration. That is good news because it gives us a stable starting point for normalization and synthetic mashup generation.


In [10]:
def resolve_other(song_dir: Path) -> Path | None:
    for name in ('other.wav', 'others.wav'):
        path = song_dir / name
        if path.exists():
            return path
    return None


def audio_info(path: Path) -> dict:
    info = sf.info(str(path))
    return {
        'samplerate': info.samplerate,
        'frames': info.frames,
        'seconds': info.frames / info.samplerate,
        'channels': info.channels,
    }


train_rows = []
for genre_dir in sorted(path for path in GENRES_STEMS.iterdir() if path.is_dir()):
    for song_dir in sorted(path for path in genre_dir.iterdir() if path.is_dir()):
        for stem_name in ('drums', 'vocals', 'bass'):
            train_rows.append(audio_info(song_dir / f'{stem_name}.wav'))
        other_path = resolve_other(song_dir)
        if other_path is not None:
            train_rows.append(audio_info(other_path))

train_audio_df = pd.DataFrame(train_rows)
display(train_audio_df[['samplerate', 'channels', 'seconds']].agg(['mean', 'min', 'max']))
print('Unique sample rates:', sorted(train_audio_df['samplerate'].unique().tolist()))
print('Unique channel counts:', sorted(train_audio_df['channels'].unique().tolist()))


,samplerate,channels,seconds
mean,44100.0,2.0,30.024047
min,44100.0,2.0,29.931973
max,44100.0,2.0,30.648889


Unique sample rates: [44100]
Unique channel counts: [2]


## 3. Train test shift

The test files look materially different from the training stems. They are mono, sampled at 22.05 kHz, and their durations vary a lot more.

In [11]:
test_df = pd.read_csv(TEST_CSV)

test_rows = []
for row in test_df.itertuples(index=False):
    rel_path = Path(row.filename)
    audio_path = DATASET_ROOT / rel_path
    info = audio_info(audio_path)
    test_rows.append(info)

test_audio_df = pd.DataFrame(test_rows)
display(test_audio_df[['samplerate', 'channels', 'seconds']].agg(['mean', 'min', 'max']))
print('Unique sample rates:', sorted(test_audio_df['samplerate'].unique().tolist()))
print('Unique channel counts:', sorted(test_audio_df['channels'].unique().tolist()))
print('Mapped test files:', len(test_audio_df))


,samplerate,channels,seconds
mean,22050.0,1.0,28.653240
min,22050.0,1.0,6.060408
max,22050.0,1.0,30.579229


Unique sample rates: [22050]
Unique channel counts: [1]
Mapped test files: 3020


## 4. Noise data

The ESC-50 noise bank is very regular: mono, 44.1 kHz, and exactly 5 seconds long. That makes it a good augmentation source, but it is much shorter than the music clips.


In [13]:
noise_paths = sorted(NOISE_DIR.glob('*.wav'))
noise_rows = [audio_info(path) for path in noise_paths]
noise_df = pd.DataFrame(noise_rows)
display(noise_df[['samplerate', 'channels', 'seconds']].agg(['mean', 'min', 'max']))
print('Noise file count:', len(noise_df))

,samplerate,channels,seconds
mean,44100.0,1.0,5.0
min,44100.0,1.0,5.0
max,44100.0,1.0,5.0


Noise file count: 2000


## Conclusions

- We should convert all train audio to one common working representation before feature extraction. A sensible default is mono audio at 22.05 kHz with fixed-duration crops.
- We should train on synthetic mashups, not just isolated stems, because the evaluation domain is mixed audio.
- We should add noise during training. The ESC-50 bank is large enough to make augmentation meaningful.
- We should enforce train-validation separation at the song level to avoid leakage across stems from the same original track.